# Eye model predictions (validation set)

Loads a trained checkpoint and shows what the network predicts on its validation eyes:
1. a grid: image | annotation | prediction | outlines over the image, with the predicted laterality
2. the predicted parameters of one eye
3. both predicted eyes pasted back onto the full face image
4. the worst / best eyes by Dice

Set `ZOOM = True` to look at random close-ups of the same eyes (as in `--zoom-eval`).
Colours: skin = tan, sclera = white, iris (+ pupil) = blue, caruncle = pink, ignored = grey.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

from eyemodel3d import train as T
import eyemodel3d.renderer.eye_model as em

warnings.filterwarnings("ignore", "Mean of empty slice")  # eyes with no foreground at all
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- edit these
CKPT = ROOT / "runs/celeb1/best.pt"
DATA = None  # None = the data the checkpoint was trained on
ZOOM = False  # True: random close-ups of the validation eyes

In [ ]:
net, space, fixed, meta = T.load_model(CKPT, dev)
H, W = meta["size"]
TAU = meta["tau_min"]
DATA = Path(DATA or meta.get("data", "/home/clement/Documents/data/Eye3D/periorbital_dataset/celeb_final_data"))
val_names = set(meta["val"])
val_pairs = [p for p in T.list_pairs(DATA) if p[0].name in val_names]
ds = T.EyeMaskDataset(val_pairs, H, W, zoom_eval=ZOOM, **meta.get("zoom", {}))
PREDICTS_LAT = "medial" not in fixed
print(f"checkpoint epoch {meta['epoch']}, val Dice {meta['val_dice']:.3f}; "
      f"{len(val_pairs)} validation images = {len(ds)} eyes; "
      f"laterality {'predicted' if PREDICTS_LAT else 'fixed (old checkpoint)'}")

In [ ]:
@torch.no_grad()
def predict(xs):
    """xs: (B, 3, H, W) normalized -> parameter values (B, P), probs (B, H, W, 4), medial (B,)."""
    z, lat = net(xs.to(dev))
    v = space.values(z)
    m = T.predicted_medial(lat, fixed)
    return v, T.render_batch(space.as_dict(v), fixed, m, H, W, TAU), m


def to_image(x):
    return (x * T.IMAGENET_STD + T.IMAGENET_MEAN).clamp(0, 1).permute(1, 2, 0).numpy()


def predict_samples(idx):
    items = [ds[int(i)] for i in idx]
    xs = torch.stack([it[0] for it in items])
    ys = torch.stack([it[1] for it in items])
    v, probs, m = predict(xs)
    return dict(
        names=[it[3] for it in items],
        lat_true=["OD" if it[2] > 0 else "OS" for it in items],
        lat_pred=["OD" if x > 0 else "OS" for x in m.tolist()],
        img=[to_image(x) for x in xs],
        gt=ys.numpy().astype(np.uint8),
        pred=probs.argmax(-1).cpu().numpy().astype(np.uint8),
        vals=v.cpu().numpy(),
        dice=T.hard_dice(probs, ys.to(dev)).cpu().numpy(),
    )


OUTLINE = {1: "w", 2: "c", 3: "m"}


def show(res):
    n = len(res["names"])
    fig, axs = plt.subplots(n, 4, figsize=(13, 2.7 * n), squeeze=False)
    for r in range(n):
        img, gt, pred = res["img"][r], res["gt"][r], res["pred"][r]
        axs[r, 0].imshow(img)
        axs[r, 1].imshow(T.colorize(gt))
        axs[r, 2].imshow(T.colorize(pred))
        axs[r, 3].imshow(img)
        for c, col in OUTLINE.items():
            if (pred == c).any():
                axs[r, 3].contour(pred == c, levels=[0.5], colors=col, linewidths=1)
        d = res["dice"][r]
        ok = "" if res["lat_true"][r] == res["lat_pred"][r] else "  WRONG SIDE"
        axs[r, 0].set_title(res["names"][r], fontsize=8)
        axs[r, 2].set_title("  ".join(f"{c[:4]} {d[i]:.2f}" for i, c in
                                      enumerate(T.TARGET_CLASSES) if i), fontsize=8)
        axs[r, 3].set_title(f"predicted {res['lat_pred'][r]}{ok}", fontsize=8,
                            color="r" if ok else "k")
        for a in axs[r]:
            a.set_axis_off()
    axs[0, 1].set_title("annotation")
    plt.tight_layout()
    plt.show()

## 1. Random validation eyes\nRe-run the cell for another draw.

In [ ]:
rng = np.random.default_rng()
res = predict_samples(rng.choice(len(ds), size=6, replace=False))
show(res)

## 2. Predicted parameters of one eye

In [ ]:
def param_table(vals):
    """Predicted values vs the checkpoint's bounds; the bar shows the position in the range
    (log scale for f). A value pinned at a bound often means the model lacks something."""
    v = torch.as_tensor(vals, device=dev)[None]
    pos = space.normalize(v)[0].cpu().numpy()
    rows = []
    for j, n in enumerate(space.names):
        k = round(20 * float(np.clip(pos[j], 0, 1)))
        rows.append(dict(param=n, value=round(float(vals[j]), 3),
                         default=round(float(space.default[j]), 3),
                         lo=round(float(space.lo[j]), 3), hi=round(float(space.hi[j]), 3),
                         position="|" + "#" * k + "." * (20 - k) + "|"))
    for n, x in meta["fixed"].items():
        rows.append(dict(param=n, value=round(x, 3), default="fixed"))
    return pd.DataFrame(rows).set_index("param")


k = 0  # row of the grid above
print(res["names"][k], "predicted", res["lat_pred"][k])
param_table(res["vals"][k])

## 3. Both eyes on the full face
Each half is predicted separately (standard framing), un-letterboxed, and pasted back onto the
original image. For old checkpoints that do not predict laterality, the right half is mirrored.

In [ ]:
@torch.no_grad()
def predict_face(img_path, mask_path=None):
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    hw = w // 2
    full = np.zeros((h, w), np.uint8)  # odd width: the middle column stays skin
    lats = []
    for side in (0, 1):
        x0 = 0 if side == 0 else w - hw
        half = img.crop((x0, 0, x0 + hw, h))
        mirror = side == 1 and not PREDICTS_LAT
        if mirror:
            half = half.transpose(Image.FLIP_LEFT_RIGHT)
        im, _ = ds.frame(half, Image.new("L", half.size, 0), None, zoom=False)
        x = (T.TF.to_tensor(im) - T.IMAGENET_MEAN) / T.IMAGENET_STD
        _, probs, m = predict(x[None])
        lats.append("OD" if m[0] > 0 else "OS")
        lbl = probs[0].argmax(-1).cpu().numpy().astype(np.uint8)
        # undo the letterbox, then the mirror
        s = min(W / hw, H / h)
        nw, nh = max(1, round(hw * s)), max(1, round(h * s))
        ox, oy = (W - nw) // 2, (H - nh) // 2
        lbl = np.array(Image.fromarray(lbl[oy:oy + nh, ox:ox + nw]).resize((hw, h), Image.NEAREST))
        full[:, x0:x0 + hw] = lbl[:, ::-1] if mirror else lbl
    gt = None
    if mask_path is not None:
        raw = np.array(Image.open(mask_path))
        gt = ds.lut[raw.astype(np.int64).clip(0, 255)]
    return np.array(img), full, gt, lats


def show_face(img_path, mask_path=None):
    img, pred, gt, lats = predict_face(img_path, mask_path)
    n = 3 if gt is not None else 2
    fig, axs = plt.subplots(1, n, figsize=(6 * n, 3))
    axs[0].imshow(img)
    for c, col in OUTLINE.items():
        if (pred == c).any():
            axs[0].contour(pred == c, levels=[0.5], colors=col, linewidths=1)
    axs[0].set_title(f"{Path(img_path).name}  predicted: image-left {lats[0]}, image-right {lats[1]}",
                     fontsize=9)
    axs[1].imshow(T.colorize(pred)); axs[1].set_title("prediction")
    if gt is not None:
        axs[2].imshow(T.colorize(gt)); axs[2].set_title("annotation")
    for a in axs:
        a.set_axis_off()
    plt.tight_layout()
    plt.show()


ip, mp = val_pairs[rng.integers(len(val_pairs))]
show_face(ip, mp)
# Any image of the same kind (no mask needed): show_face("path/to/face_crop.jpg")

## 4. Worst and best eyes (mean foreground Dice), laterality accuracy

In [ ]:
scores, lat_ok = [], []
B = 64
for s in range(0, len(ds), B):
    r = predict_samples(range(s, min(s + B, len(ds))))
    scores += list(np.nanmean(r["dice"][:, 1:], 1))
    lat_ok += [a == b for a, b in zip(r["lat_true"], r["lat_pred"])]
scores = np.array(scores)
order = np.argsort(scores)
print(f"mean foreground Dice {np.nanmean(scores):.3f}   median {np.nanmedian(scores):.3f}   "
      f"laterality accuracy {np.mean(lat_ok):.3f}")
plt.figure(figsize=(6, 2.5)); plt.hist(scores, bins=30); plt.xlabel("mean foreground Dice"); plt.show()

In [ ]:
print("worst"); show(predict_samples(order[:5]))

In [ ]:
print("best"); show(predict_samples(order[-5:][::-1]))